## RAG Ingestion Pipeline

Ingestion ==>  Chunking ==> Embedding ==> Vector Store

In [35]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [36]:
def process_all_pdfs(pdf_directory):
    all_documents = []

    pdf_files = list(filter(lambda f: f.endswith('.pdf'), os.listdir(pdf_directory)))
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}.")


    for filename in pdf_files:

        pdf_path = os.path.join(pdf_directory, filename)
        print(f"\n Processing {filename} with {len(filename)} page(s).")

        try:
            loader = PyPDFLoader(pdf_path)
            documents = loader.load()

            for doc in documents:
                doc.metadata["source"] = filename
                doc.metadata["total_pages"] = len(documents)
                doc.metadata["file_size"] = os.path.getsize(pdf_path)
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Successfully loaded {len(documents)} page(s) from {filename} using PyPDFLoader.")

        except Exception as e:
            print(f"Failed to load {filename} with PyPDFLoader: {e}. ")

    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

            
all_pdf_documents = process_all_pdfs("../data/pdf_files")

Found 11 PDF files in ../data/pdf_files.

 Processing acceptable_use_policy.pdf with 25 page(s).
Successfully loaded 1 page(s) from acceptable_use_policy.pdf using PyPDFLoader.

 Processing business_continuity_policy.pdf with 30 page(s).
Successfully loaded 1 page(s) from business_continuity_policy.pdf using PyPDFLoader.

 Processing code_of_conduct.pdf with 19 page(s).
Successfully loaded 1 page(s) from code_of_conduct.pdf using PyPDFLoader.

 Processing company_overview.pdf with 20 page(s).
Successfully loaded 1 page(s) from company_overview.pdf using PyPDFLoader.

 Processing diversity_equity_inclusion_policy.pdf with 37 page(s).
Successfully loaded 1 page(s) from diversity_equity_inclusion_policy.pdf using PyPDFLoader.

 Processing hr_policies_handbook.pdf with 24 page(s).
Successfully loaded 1 page(s) from hr_policies_handbook.pdf using PyPDFLoader.

 Processing incident_response_policy.pdf with 28 page(s).
Successfully loaded 1 page(s) from incident_response_policy.pdf using PyPD

In [37]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'acceptable_use_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_size': 2390, 'file_type': 'pdf'}, page_content='Northstar Meridian Group\nAcceptable Use Policy\nDocument ID: IT-AUP-006   |   Owner: IT Operations   |   Version: 2.0   |   Effective: 2026-07-26\nCompany devices and accounts are provided for business use.\nUsers must:\n- Keep devices updated and locked when unattended\n- Use only approved software and cloud services\n- Avoid installing unlicensed or malicious software\n- Store company files in approved repositories\n- Report lost devices, phishing, or suspicious activity promptly\nUsers must not:\n- Circumvent security controls or monitoring\n- Share corporate accounts or bypass access reviews\n- Download copyright-protected material without permission\n- Use resources for gambling, harassment, or other inappropriate content\nInternal use only | Internal Knowledge 

In [38]:
## Text splitting into chunks 
def split_documents(documents, chunk_size=500, chunk_overlap=20):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    return split_docs

In [39]:
chunks = split_documents(all_pdf_documents)
chunks

Split 11 documents into 23 chunks.


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'acceptable_use_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_size': 2390, 'file_type': 'pdf'}, page_content='Northstar Meridian Group\nAcceptable Use Policy\nDocument ID: IT-AUP-006   |   Owner: IT Operations   |   Version: 2.0   |   Effective: 2026-07-26\nCompany devices and accounts are provided for business use.\nUsers must:\n- Keep devices updated and locked when unattended\n- Use only approved software and cloud services\n- Avoid installing unlicensed or malicious software\n- Store company files in approved repositories\n- Report lost devices, phishing, or suspicious activity promptly\nUsers must not:'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'acceptable_use_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_size': 2390, 'file_type': 'pdf'}, page_content='Users must not:\n- Circumvent security controls 

## Embedding and Vector Store BD

In [40]:
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [41]:
class EmbeddingManager:

    "Handles document embeddings using SentenceTransformer"

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name= model_name
        self.model = None
        self._load_model()

    def _load_model(self):

        "This method loads the SentenceTransformer model."

        try:

            self.model = SentenceTransformer(self.model_name)
            print(f"Successfully loaded model: {self.model_name}. Embedding dimension: {self.model.get_embedding_dimension()}")

        except Exception as e:
            print(f"Failed to load model: {self.model_name}. Error: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:

        """
        Generate embeddings for a list of texts.

        Args:
            texts (List[str]): List of text strings to embed.

        Returns:
            np.ndarray: Array of embeddings.
        
        """

        if not self.model:
            raise ValueError("Model is not loaded. Call _load_model() first.")

        print(f"Generating embeddings for {len(texts)} texts using model: {self.model_name}.")

        try:
            embeddings = self.model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
            print(f"Successfully generated embeddings. Shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Failed to generate embeddings. Error: {e}")
            raise

embedding_manager = EmbeddingManager()
embedding_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6808.51it/s]


Successfully loaded model: all-MiniLM-L6-v2. Embedding dimension: 384


In [42]:
## Vector Store

class VectorStore:

    "Handles storage and retrieval of document embeddings using ChromaDB"

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """
        Initializes the ChromaDB client and collection.
        """        
        try:

            #create persistent chroma db directory if it doesn't exist
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #get or create collection
            # A collection is a group of documents and their embeddings. 
            # If the collection already exists, it will be retrieved; otherwise, a new collection will be created.
            self.collection = self.client.get_or_create_collection(name=self.collection_name,
                                                                   metadata={"description": "PDF embeddings for RAG"}
                                                                   )
            print(f"Vector store initialized. collection: {self.collection_name}, persist_directory: {self.persist_directory}")
            print(f"Existing document count in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store. Error: {e}")
            raise

    #add documents and their embeddings to the collection
    def add_documents(self, documents: List[Dict[str, Any]], embeddings: np.ndarray):
        """
        Adds documents and their embeddings to the ChromaDB collection.

        Args:
            documents (List[Dict[str, Any]]): List of document objects.
            embeddings (np.ndarray): Array of embeddings corresponding to the documents.
        """

        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents must match number of embeddings.")

        ids = [str(uuid.uuid4()) for _ in range(len(documents))]
        metadatas = [{"source": doc.metadata["source"], 
                      "total_pages": doc.metadata["total_pages"], 
                      "file_size": doc.metadata["file_size"], 
                      "file_type": doc.metadata["file_type"],
                      "content_length": len(doc.page_content)} 
                      for doc in documents]
        documents_content = [doc.page_content for doc in documents]

        #add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings.tolist(),
                metadatas=metadatas,
                documents=documents_content
            )
            print(f"Successfully added {len(documents)} documents to the collection: {self.collection_name}.")
            print(f"Total document count in collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Failed to add documents to the collection. Error: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. collection: pdf_documents, persist_directory: ../data/vector_store
Existing document count in collection: 0


In [43]:
### convert all chunks to embeddings and add to vector store

texts = [doc.page_content for doc in chunks]

 ### generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

print(type(embeddings))
print(type(chunks))

### store in vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 23 texts using model: all-MiniLM-L6-v2.


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.79it/s]

Successfully generated embeddings. Shape: (23, 384)
<class 'numpy.ndarray'>
<class 'list'>
Successfully added 23 documents to the collection: pdf_documents.
Total document count in collection after addition: 23


## Retrieval Pipeline from Vector store

In [ ]:
class QueryEngine:

    "Handles querying the vector store and retrieving relevant documents."

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager


    def query(self, query_text: str, top_k: int = 3, score_threshold: float = 0.0) -> List[Dict[str, Any]]:

        """
        Queries the vector store for the most relevant documents to the input query.

        Args:
            query_text (str): The input query text.
            top_k (int): The number of top relevant documents to retrieve.
            score_threshold (float): The minimum similarity score for a document to be considered relevant.
        """
        #generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query_text])[0]

        #search the vector store for the most similar documents
        try:
            results = self.vector_store.collection.query(
                query_embeddings=query_embedding.tolist(),
                n_results=top_k,
                include=["documents", "metadatas", "distances"]
            )

            #filter results based on score threshold
            filtered_results = []

            for doc, doc_id, metadata, distance in zip(results["documents"][0], results["ids"][0], results["metadatas"][0], results["distances"][0]):
                similarity_score = 1 - distance  # Convert distance to similarity
                if similarity_score >= score_threshold:
                    filtered_results.append({
                        "id": doc_id,
                        "document": doc,
                        "metadata": metadata,
                        "similarity_score": similarity_score,
                        "distance": distance,
                        "rank": len(filtered_results) + 1  # Rank based on the order of appearance
                    })

            if filtered_results:
                print(f"Found {len(filtered_results)} relevant documents for the query: {query_text}")
            else:
                print(f"No results found for the query: {query_text}")

            return filtered_results

        except Exception as e:
            print(f"Failed to query the vector store. Error: {e}")
            raise   

rag_retriever = QueryEngine(vectorstore, embedding_manager)

In [54]:
rag_retriever.query("Northstar Meridian Group?", top_k=10, score_threshold=0.0)

Generating embeddings for 1 texts using model: all-MiniLM-L6-v2.


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.32it/s]

Successfully generated embeddings. Shape: (1, 384)
Found 2 relevant documents for the query: Northstar Meridian Group?


[{'id': '94b161f3-c028-4c14-a86c-0140ab687159',
  'document': 'Northstar Meridian Group\nCompany Overview and Profile\nDocument ID: CORP-OVR-001   |   Owner: Corporate Strategy Office   |   Version: 1.0   |   Effective: 2026-07-26\nNorthstar Meridian Group is a fictional enterprise used to populate the internal\nknowledge assistant demo dataset.\nHeadquarters: Chicago, Illinois, USA\nIndustry: Business software and managed services\nEmployee count: Approximately 420 employees\nWebsite: https://northstarmeridian.example\nBusiness focus:',
  'metadata': {'file_type': 'pdf',
   'file_size': 2678,
   'source': 'company_overview.pdf',
   'content_length': 469,
   'total_pages': 1},
  'similarity_score': 0.2842472195625305,
  'distance': 0.7157527804374695,
  'rank': 1},
 {'id': 'a838fd2e-90c6-4ff1-aa2f-b8d3a9122284',
  'document': 'Northstar Meridian Group\nDiversity, Equity, and Inclusion Policy\nDocument ID: HR-DEI-003   |   Owner: People Operations   |   Version: 1.2   |   Effective: 202